Codigo adaptado que da la longitud de las peticiones

In [9]:
# -*- coding: utf-8 -*-
"""
Created on Fri 02/10/2024

@author: Adapted by Carlos Javier Navarro
"""

import json
from openai import OpenAI
import pandas as pd
from IPython.display import Image, display

OPENAI_API_KEY='sk-svcacct-jLSnF1FIehnmGBvWCayuT3BlbkFJ1ARy381duuMQxcc5ejUH'

client = OpenAI(api_key=OPENAI_API_KEY)

dataset_path = "listado_imagenes2.csv" 
df = pd.read_csv(dataset_path)
df.head()


# Rhoaifa_new prompt 1
caption_system_prompt = '''
Classify the image into one of these categories: Cultural_Religious, Fauna_Flora, Gastronomy, Nature, Sports, or Urban_Rural, I just need you to give me the name of the class as a result without any extra explanation. 
'''


#Rohaifa Simple prompt
# caption_system_prompt = '''
# Classify the image into one of the classes: Cultural_Religious, Fauna_Flora, Gastronomy, Nature, Sports, or Urban_Rural
# '''



# Crear tareas para el procesamiento por lotes
tasks = []

for index, row in df.iterrows():
    title = row['id2']
    img_url = row['url']
    
    user_message = img_url
    print(f"Message Length for task {index}: {len(user_message)} tokens")  # Imprime la longitud del mensaje

    task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-mini",
            "temperature": 0.2,
            "max_tokens": 300,  
            "messages": [
                {
                    "role": "system",
                    "content": caption_system_prompt  # Solo el prompt del sistema y el mensaje del usuario
                },
                {
                    "role": "user",
                    "content": user_message,  # Usa el mensaje combinado sin historial
                },
            ]            
        }
    }
    
    tasks.append(task)
 

# Crear y subir el archivo
file_name = "batch_tasks_earthcul_Rohaifa_gtp.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

# Uploading the file
batch_file = client.files.create(
    file=open(file_name, "rb"),
    purpose="batch"
)



Message Length for task 0: 85 tokens
Message Length for task 1: 85 tokens
Message Length for task 2: 85 tokens
Message Length for task 3: 85 tokens


In [10]:
print(batch_file)

FileObject(id='file-yNcQrcJbLmQXvRLKJvujSKzG', bytes=2144, created_at=1727947147, filename='batch_tasks_earthcul_Rohaifa_gtp.jsonl', object='file', purpose='batch', status='processed', status_details=None)


In [11]:
# Creating the batch job
batch_job = client.batches.create(
  input_file_id=batch_file.id,
  endpoint="/v1/chat/completions",
  completion_window="24h"
)

In [2]:
from openai import OpenAI
OPENAI_API_KEY='sk-svcacct-jLSnF1FIehnmGBvWCayuT3BlbkFJ1ARy381duuMQxcc5ejUH'
client = OpenAI(api_key=OPENAI_API_KEY)

batch_id = 'batch_66fd6d376bd8819099b920a36e5803dc' #batch_job.id
## Checking batch status
batch_job = client.batches.retrieve(batch_id)
print(batch_job)

Batch(id='batch_66fd6d376bd8819099b920a36e5803dc', completion_window='24h', created_at=1727884599, endpoint='/v1/chat/completions', input_file_id='file-SvsmKbXCt0Hc3DUvXHK6iCET', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1727888800, error_file_id=None, errors=None, expired_at=None, expires_at=1727970999, failed_at=None, finalizing_at=1727888752, in_progress_at=1727884600, metadata=None, output_file_id='file-vlPSewIzz2zhg75MNzYhhHHJ', request_counts=BatchRequestCounts(completed=672, failed=0, total=672))


In [3]:
# Retrieving results
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content

In [4]:

result_file_name = "test.jsonl"

with open(result_file_name, 'wb') as file:
    file.write(result)

In [5]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [8]:
import pandas as pd
from IPython.display import Image, display

# Reading only the first results
for res in results[:5]:
    task_id = res['custom_id']
    # Getting index from task id
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    item = df.iloc[int(index)]
    img_url = item['id']
    img = Image(url=img_url)
    display(img)
    print(f"CAPTION:{result}\n\n {img_url}")

CAPTION:Fauna_Flora

 batch_req_66fd7d70b98881908f46d095d7ac66ac


CAPTION:Fauna_Flora

 batch_req_66fd7d70cbbc819090f592b122131cd8


CAPTION:Fauna_Flora

 batch_req_66fd7d70dc188190ae9eb845b0a11c8d


CAPTION:Fauna_Flora

 batch_req_66fd7d7107d88190961b4719e1ec35ed


CAPTION:Fauna_Flora

 batch_req_66fd7d7117188190b9b9e3673e472abe


Extraccion de resultados almacenados en content en el caso de una clase usando el script original de Rohaifa

In [30]:
import pandas as pd
import json
import re

# Ruta del archivo JSONL
# input_file = r"H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\batch results\batch_66f2c3bc5f188190a562d7ec05216f9d_output_RK.jsonl"

input_file = r"C:\Users\carlo\Downloads\prompt2.jsonl"

# Leer el archivo JSONL
data = []
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

# Convertir a DataFrame
df = pd.json_normalize(data)

# Definir las categorías posibles
categories = ["Cultural_Religious", "Fauna_Flora", "Gastronomy", "Nature", "Sports", "Urban_Rural"]

def get_class(row):
    content = row['response.body.choices'][0]['message']['content']
    
    # Buscar cualquier categoría dentro del contenido
    for category in categories:
        if category in content:
            return category
    
    # Si no encuentra ninguna coincidencia, devuelve None
    return None

# Extraer la clase
df['Class'] = df.apply(get_class, axis=1)

# Guardar como CSV
output_file = r"H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\batch results\test2.csv"
df.to_csv(output_file, index=False)

print(f"Archivo CSV guardado en: {output_file}")


Archivo CSV guardado en: H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\batch results\test2.csv


**Extraccion usando el prompt 1:**

```python
caption_system_prompt = '''
I have an image that I would like to classify into one of the following categories: Cultural_Religious, Fauna_Flora, Gastronomy, Nature, Sports, or Urban_Rural. Please analyze the contents of the image and, based on its dominant features, tell me which category it fits best.
'''
```


In [1]:
import pandas as pd
import json
import re

# Ruta del archivo JSONL
input_file = r"H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\Train_Data\batch results\batch_66fd6d376bd8819099b920a36e5803dc_output.jsonl"

# Leer el archivo JSONL
data = []
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

# Convertir a DataFrame
df = pd.json_normalize(data)

# Definir las categorías posibles
categories = ["Cultural_Religious", "Fauna_Flora", "Gastronomy", "Nature", "Sports", "Urban_Rural"]

def get_class(row):
    content = row['response.body.choices'][0]['message']['content']
    
    # Buscar cualquier categoría dentro del contenido
    for category in categories:
        if category in content:
            return category
    
    # Si no encuentra ninguna coincidencia, devuelve None
    return None

# Extraer la clase
df['Class'] = df.apply(get_class, axis=1)

# Guardar como CSV
output_file = r"H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\Train_Data\batch results\train_data2.csv"
df.to_csv(output_file, index=False)

print(f"Archivo CSV guardado en: {output_file}")

Archivo CSV guardado en: H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\Train_Data\batch results\train_data2.csv


**Extraccion usando el prompt 2:** 

caption_system_prompt = '''
I have an image that I would like to classify into one of the following categories: Cultural_Religious, Fauna_Flora, Gastronomy, Nature, Sports, or Urban_Rural. Please analyze the contents of the image and, based on its dominant features, tell me which category it fits best. The definitions for each category are as follows:
Cultural_Religious: The image depicts religious symbols, cultural artifacts, traditions, ceremonies, or anything related to culture and belief systems.
Fauna_Flora: The image features animals (fauna) or plants (flora) in any environment.
Gastronomy: The image is related to food, cooking, culinary experiences, or dining.
Nature: The image contains natural landscapes, such as mountains, rivers, forests, or other untouched environments.
Sports: The image shows physical activities, competitions, or sports equipment related to athletic endeavors.
Urban_Rural: The image captures cityscapes, villages, rural settings, buildings, or any human-made environments.
'''

In [25]:
import pandas as pd
import json
import re

# Ruta del archivo JSONL
input_file = r"H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\batch results\batch_YXy6D8op4M3fVBfCb5UlKFHi_output_GTP_def.jsonl"

# Leer el archivo JSONL
data = []
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

# Convertir a DataFrame
df = pd.json_normalize(data)

# Definir las categorías posibles
categories = ["Cultural_Religious", "Fauna_Flora", "Gastronomy", "Nature", "Sports", "Urban_Rural"]

def get_class(row):
    content = row['response.body.choices'][0]['message']['content']
    
    # Buscar cualquier categoría dentro del contenido
    for category in categories:
        if category in content:
            return category
    
    # Si no encuentra ninguna coincidencia, devuelve None
    return None

# Extraer la clase
df['Class'] = df.apply(get_class, axis=1)

# Guardar como CSV
output_file = r"H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\batch results\output_gtp_def.csv"
df.to_csv(output_file, index=False)

print(f"Archivo CSV guardado en: {output_file}")

Archivo CSV guardado en: H:\Mi unidad\2024\CHAT GTP for R\Rohaifa Clusters vs ChatGTP\batch results\output_gtp_def.csv
